# 1. Khám Phá Dữ Liệu Yellow Tripdata (2026-01)


In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

file_path = r"../../data/yellow_tripdata_2026-01.parquet"

print('Đang tải dữ liệu vào bộ nhớ (có thể mất khoảng 5-10 giây)...')
df = pd.read_parquet(file_path)

print(f'Tải thành công! Tổng số dòng: {len(df):,}')
print(f'Kích thước bộ nhớ sử dụng: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
df.head()

Đang tải dữ liệu vào bộ nhớ (có thể mất khoảng 5-10 giây)...
Tải thành công! Tổng số dòng: 3,724,889
Kích thước bộ nhớ sử dụng: 647.96 MB


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [2]:
# Xem thông tin tổng quan về các cột, kiểu dữ liệu và số lượng giá trị không bị thiếu (non-null)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

## 1.1. Tích hợp dữ liệu khu vực, nhà cung cấp và mã biểu giá & Feature Engineering
Kết nối với `taxi_zone_lookup.csv` để lấy tên quận (Borough) và khu vực (Zone), thêm cột ratecode_name. Tạo thêm các biến thời gian.

In [3]:
import pandas as pd

# 0. DỌN DẸP CỘT CŨ (Tránh lỗi KeyError khi vô tình bấm chạy cell này nhiều lần)
cols_to_drop = [c for c in df.columns if c.startswith('pu_') or c.startswith('do_') or c in ['vendor_name', 'ratecode_name', 'pickup_dt', 'hour_of_day', 'day_of_week']]
df = df.drop(columns=cols_to_drop, errors='ignore')

# 1. ĐỌC VÀ MERGE KHU VỰC
lookup_path = r'../../data/taxi_zone_lookup.csv'
lookup = pd.read_csv(lookup_path)

# Merge pickup location
df = df.merge(lookup.add_prefix('pu_'), left_on='PULocationID', right_on='pu_LocationID', how='left')
# Merge drop-off location
df = df.merge(lookup.add_prefix('do_'), left_on='DOLocationID', right_on='do_LocationID', how='left')

# 2. ÁNH XẠ VENDOR_ID VÀ RATECODE_ID
vendor_map = {1: 'Creative Mobile', 2: 'Curb Mobility', 6: 'Myle Technologies', 7: 'Helix'}
df['vendor_name'] = df['VendorID'].map(vendor_map)

ratecode_map = {
    1.0: 'Standard rate',
    2.0: 'JFK',
    3.0: 'Newark',
    4.0: 'Nassau or Westchester',
    5.0: 'Negotiated fare',
    6.0: 'Group ride',
    99.0: 'Null/unknown'
}
df['ratecode_name'] = df['RatecodeID'].map(ratecode_map)

# 3. FEATURE ENGINEERING THỜI GIAN
df['pickup_dt'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['hour_of_day'] = df['pickup_dt'].dt.hour
df['day_of_week'] = df['pickup_dt'].dt.dayofweek

# 4. HIỂN THỊ KẾT QUẢ
df[['pu_Borough', 'do_Borough', 'vendor_name', 'hour_of_day', 'day_of_week','RatecodeID', 'ratecode_name']].head()


,pu_Borough,do_Borough,vendor_name,hour_of_day,day_of_week,RatecodeID,ratecode_name
0,Manhattan,Manhattan,Curb Mobility,0,3,1.0,Standard rate
1,Manhattan,Manhattan,Creative Mobile,0,3,1.0,Standard rate
2,Manhattan,Manhattan,Creative Mobile,0,3,1.0,Standard rate
3,Manhattan,Manhattan,Curb Mobility,0,3,1.0,Standard rate
4,Manhattan,Manhattan,Curb Mobility,0,3,1.0,Standard rate


## 1.2. Kiểm tra chất lượng dữ liệu

### Kiểm tra mising value


In [4]:
import pandas as pd

# Tạo một danh sách rỗng để chứa dữ liệu tổng kết
summary_data = []

# Quét qua từng cột trong DataFrame
for col in df.columns:
    # 1. Kiểm tra Missing Values
    missing_count = df[col].isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    
    # 2. Đếm số lượng giá trị khác nhau (Unique)
    unique_count = df[col].nunique()
    
    # 3. Kiểm tra Range (Min, Max) cho các biến dạng Số hoặc Datetime
    # Bỏ qua Min/Max đối với biến dạng chuỗi (String/Object) để tránh lỗi
    if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):
        min_val = df[col].min()
        max_val = df[col].max()
    else:
        min_val = '-'
        max_val = '-'
        
    # Thêm thông tin của cột hiện tại vào danh sách
    summary_data.append({
        'Tên cột (Feature)': col,
        'Kiểu dữ liệu': df[col].dtype,
        'Số lượng Missing': missing_count,
        'Tỷ lệ Missing (%)': round(missing_pct, 2),
        'Số lượng Unique': unique_count,
        'Giá trị Min': min_val,
        'Giá trị Max': max_val
    })

# Chuyển đổi danh sách thành một DataFrame mới để hiển thị cho đẹp
df_summary = pd.DataFrame(summary_data)

# Sắp xếp bảng theo Tỷ lệ Missing giảm dần (cột nào thiếu nhiều nhất sẽ nổi lên trên)
df_summary = df_summary.sort_values(by='Tỷ lệ Missing (%)', ascending=False).reset_index(drop=True)

# Hiển thị kết quả (Bạn có thể dùng style.background_gradient để bảng có màu sắc sinh động hơn)
df_summary.style.background_gradient(cmap='Reds', subset=['Tỷ lệ Missing (%)'])



,Tên cột (Feature),Kiểu dữ liệu,Số lượng Missing,Tỷ lệ Missing (%),Số lượng Unique,Giá trị Min,Giá trị Max
0,passenger_count,float64,1088058,29.210000,10,0.000000,9.000000
1,Airport_fee,float64,1088058,29.210000,8,-1.750000,26.750000
2,RatecodeID,float64,1088058,29.210000,7,1.000000,99.000000
3,store_and_fwd_flag,object,1088058,29.210000,2,-,-
4,congestion_surcharge,float64,1088058,29.210000,3,-2.500000,2.500000
5,ratecode_name,object,1088058,29.210000,7,-,-
6,do_service_zone,object,22302,0.600000,4,-,-
7,do_Borough,object,16286,0.440000,7,-,-
8,do_Zone,object,6016,0.160000,258,-,-
9,pu_service_zone,object,5930,0.160000,4,-,-


### Kiểm tra dữ liệu trùng lặp và tính chính xác dữ liệu (tính chính xác về thời gian, các biến < 0 bất thường>)


In [5]:
# 1. KIỂM TRA DỮ LIỆU TRÙNG LẶP (Duplicate Records)
try:
    dup_count = df.duplicated(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID']).sum()
except:
    dup_count = 0
dup_pct = (dup_count / len(df)) * 100
print("--- 1. DỮ LIỆU TRÙNG LẶP ---")
print(f"Số dòng trùng lặp hoàn toàn: {dup_count:,} dòng ({dup_pct:.4f}%)\n")


# 2. KIỂM TRA CÁC GIÁ TRỊ BẤT THƯỜNG (Invalid Values)
print("--- 2. GIÁ TRỊ BẤT THƯỜNG / SAI LOGIC ---")

# a. Số lượng hành khách (passenger_count) <= 0
invalid_passengers = df[df['passenger_count'] <= 0].shape[0]
print(f"• Số chuyến xe không có hành khách (<= 0): {invalid_passengers:,} chuyến")

# b. Quãng đường (trip_distance) <= 0
invalid_distance = df[df['trip_distance'] <= 0].shape[0]
print(f"• Số chuyến xe có quãng đường <= 0 dặm: {invalid_distance:,} chuyến")

# c. Chi phí chuyến đi (fare_amount) bị âm hoặc bằng 0
# Lưu ý: Tiền cước thường có giá trị âm trong hệ thống TLC khi chuyến đi bị hủy, tranh chấp hoặc hoàn tiền (refund/dispute).
invalid_fare = df[df['fare_amount'] <= 0].shape[0]
print(f"• Số chuyến xe có tiền cước thuần (fare) <= 0 USD: {invalid_fare:,} chuyến")

# d. Thời gian chuyến đi bị âm (Drop-off diễn ra TRƯỚC Pick-up)
invalid_time = df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']].shape[0]
print(f"• Số chuyến xe có thời gian trả khách diễn ra TRƯỚC thời gian đón (Lỗi đồng hồ): {invalid_time:,} chuyến")

# e. Mã RatecodeID không hợp lệ (Không nằm trong chuẩn 1, 2, 3, 4, 5, 6, 99)
valid_ratecodes = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 99.0]
invalid_ratecode = df[~df['RatecodeID'].isin(valid_ratecodes) & df['RatecodeID'].notnull()].shape[0]
print(f"• Số chuyến xe chứa mã RatecodeID lạ (ngoài từ điển): {invalid_ratecode:,} chuyến")


--- 1. DỮ LIỆU TRÙNG LẶP ---
Số dòng trùng lặp hoàn toàn: 35,723 dòng (0.9590%)

--- 2. GIÁ TRỊ BẤT THƯỜNG / SAI LOGIC ---
• Số chuyến xe không có hành khách (<= 0): 14,787 chuyến
• Số chuyến xe có quãng đường <= 0 dặm: 125,738 chuyến
• Số chuyến xe có tiền cước thuần (fare) <= 0 USD: 41,545 chuyến
• Số chuyến xe có thời gian trả khách diễn ra TRƯỚC thời gian đón (Lỗi đồng hồ): 1 chuyến
• Số chuyến xe chứa mã RatecodeID lạ (ngoài từ điển): 0 chuyến


In [6]:
# Danh sách tất cả các cột liên quan đến tiền
money_columns = [
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 
    'improvement_surcharge', 'total_amount', 'congestion_surcharge', 
    'Airport_fee', 'cbd_congestion_fee'
]

print("--- KIỂM TRA LỖI TIỀN ÂM (< 0) CHO TẤT CẢ CÁC CỘT CHI PHÍ ---")
for col in money_columns:
    if col in df.columns:
        negative_count = df[df[col] < 0].shape[0]
        if negative_count > 0:
            print(f"• Cột '{col}' có {negative_count:,} dòng bị âm (< 0)")

print("\n--- KIỂM TRA LỖI CƯỚC HOẶC TỔNG TIỀN BẰNG 0 (<= 0) ---")
invalid_fare = df[df['fare_amount'] <= 0].shape[0]
invalid_total = df[df['total_amount'] <= 0].shape[0]

print(f"• Số chuyến xe có fare_amount <= 0: {invalid_fare:,} chuyến")
print(f"• Số chuyến xe có total_amount <= 0: {invalid_total:,} chuyến")

# => KẾT LUẬN LÀM SẠCH (DATA CLEANING):
# Bạn có thể dùng dòng lệnh sau để giữ lại các dữ liệu SẠCH:
# df_clean = df[(df['fare_amount'] > 0) & (df['total_amount'] > 0) & (df['trip_distance'] > 0) & (df['passenger_count'] > 0)]


--- KIỂM TRA LỖI TIỀN ÂM (< 0) CHO TẤT CẢ CÁC CỘT CHI PHÍ ---
• Cột 'fare_amount' có 39,463 dòng bị âm (< 0)
• Cột 'extra' có 20,035 dòng bị âm (< 0)
• Cột 'mta_tax' có 37,730 dòng bị âm (< 0)
• Cột 'tip_amount' có 67 dòng bị âm (< 0)
• Cột 'tolls_amount' có 3,391 dòng bị âm (< 0)
• Cột 'improvement_surcharge' có 39,732 dòng bị âm (< 0)
• Cột 'total_amount' có 39,984 dòng bị âm (< 0)
• Cột 'congestion_surcharge' có 30,448 dòng bị âm (< 0)
• Cột 'Airport_fee' có 9,225 dòng bị âm (< 0)
• Cột 'cbd_congestion_fee' có 25,197 dòng bị âm (< 0)

--- KIỂM TRA LỖI CƯỚC HOẶC TỔNG TIỀN BẰNG 0 (<= 0) ---
• Số chuyến xe có fare_amount <= 0: 41,545 chuyến
• Số chuyến xe có total_amount <= 0: 40,417 chuyến


In [7]:
# Đảm bảo cột pickup_dt đã được chuyển sang dạng datetime
if 'pickup_dt' not in df.columns:
    df['pickup_dt'] = pd.to_datetime(df['tpep_pickup_datetime'])

# 3. KIỂM TRA LỖI THỜI GIAN NGOÀI PHẠM VI (Out-of-bound Dates)
print("--- 3. LỖI THỜI GIAN (OUT-OF-BOUND) ---")

# Giới hạn mốc thời gian chuẩn của file: Tháng 01/2026
start_date = pd.to_datetime('2026-01-01')
end_date = pd.to_datetime('2026-02-01')

# Lọc các chuyến bắt đầu trước tháng 1/2026
past_trips = df[df['pickup_dt'] < start_date]

# Phân loại thêm: Đêm Giao Thừa (Hợp lý) vs Lỗi đồng hồ (Rác)
nye_trips = past_trips[past_trips['pickup_dt'] >= pd.to_datetime('2025-12-31')]
error_past_trips = past_trips[past_trips['pickup_dt'] < pd.to_datetime('2025-12-31')]

# Các chuyến bắt đầu sau tháng 1/2026 (Chuyến xe đến từ tương lai do lỗi)
future_trips = df[df['pickup_dt'] >= end_date]

print(f"• Tổng số chuyến xe nằm trước tháng 01/2026: {len(past_trips):,} chuyến")
print(f"  -> Trong đó hợp lệ (Chạy vắt qua Đêm Giao Thừa 31/12): {len(nye_trips):,} chuyến")
print(f"  -> Trong đó LỖI ĐỒNG HỒ (Tồn tại từ trước 31/12/2025): {len(error_past_trips):,} chuyến")
print(f"• Tổng số chuyến xe đến từ tương lai (Sau 31/01/2026): {len(future_trips):,} chuyến")

# Cho phép hiển thị tối đa 50 cột trên màn hình (để không bị ẩn mất cột nào)
pd.set_option('display.max_columns', 50)


# Xóa cột pickup_dt vì đã lấy xong Giờ và Ngày
df = df.drop(columns=['pickup_dt'])

# In ra chuyến xe đến từ tương lai dưới dạng hàng ngang
future_trips


--- 3. LỖI THỜI GIAN (OUT-OF-BOUND) ---
• Tổng số chuyến xe nằm trước tháng 01/2026: 6 chuyến
  -> Trong đó hợp lệ (Chạy vắt qua Đêm Giao Thừa 31/12): 6 chuyến
  -> Trong đó LỖI ĐỒNG HỒ (Tồn tại từ trước 31/12/2025): 0 chuyến
• Tổng số chuyến xe đến từ tương lai (Sau 31/01/2026): 1 chuyến


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pu_LocationID,pu_Borough,pu_Zone,pu_service_zone,do_LocationID,do_Borough,do_Zone,do_service_zone,vendor_name,ratecode_name,pickup_dt,hour_of_day,day_of_week
2635466,2,2026-02-01 00:45:01,2026-02-01 01:03:08,1.0,1.78,1.0,N,79,249,1,15.6,1.0,0.5,5.34,0.0,1.0,26.69,2.5,0.0,0.75,79,Manhattan,East Village,Yellow Zone,249,Manhattan,West Village,Yellow Zone,Curb Mobility,Standard rate,2026-02-01 00:45:01,0,6


### Kiểm tra tính logic dữ liệu

In [8]:
import numpy as np

print("--- KIỂM TRA GIÁ TRỊ NGOẠI LAI (OUTLIERS) ---")

# 1. TÍNH THỜI GIAN VÀ VẬN TỐC
# Đảm bảo cột dropoff là datetime (Cột pickup bạn đã có ở bước trước)
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

# Thời gian đi = (Thả khách - Đón khách) đổi ra Phút
df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

# Tính vận tốc = Quãng đường / (Thời gian / 60) 
# (Dùng np.where để tránh lỗi chia cho 0 nếu thời gian = 0 phút)
df['speed_mph'] = np.where(
    df['trip_duration_minutes'] > 0, 
    df['trip_distance'] / (df['trip_duration_minutes'] / 60), 
    np.nan
)

# Thống kê ngoại lai Vận tốc & Thời gian
speed_too_fast = df[df['speed_mph'] > 100].shape[0]
duration_too_long = df[df['trip_duration_minutes'] > 24 * 60].shape[0] # Chạy hơn 24 tiếng

print(f"• Số chuyến xe 'bay' với tốc độ > 100 dặm/giờ (Phi lý): {speed_too_fast:,} chuyến")
print(f"• Số chuyến xe chạy liên tục trên 24 tiếng (Lỗi quên tắt đồng hồ): {duration_too_long:,} chuyến")

# 2. KIỂM TRA TIỀN CƯỚC (FARE_AMOUNT) TRÊN TRỜI
# Tìm khoảng cước phí phổ biến nhất bằng Tứ phân vị (IQR)
Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR # Dùng 3*IQR cho khoảng sai số lỏng hơn một chút

crazy_fares = df[df['fare_amount'] > 300].shape[0]

print(f"\n• Mức cước trần (Upper bound theo chuẩn 3-IQR) là khoảng: ${upper_bound:.2f}")
print(f"• Số chuyến xe có cước phí siêu đắt (Hơn $300): {crazy_fares:,} chuyến")

# Xem tận mắt TOP 5 cuốc xe "chặt chém" nhất
print("\n--- TOP 5 CHUYẾN XE CÓ CƯỚC ĐẮT NHẤT ---")
df[['tpep_pickup_datetime', 'trip_distance', 'trip_duration_minutes', 'speed_mph', 'fare_amount', 'total_amount']].sort_values(by='fare_amount', ascending=False).head(5)


--- KIỂM TRA GIÁ TRỊ NGOẠI LAI (OUTLIERS) ---
• Số chuyến xe 'bay' với tốc độ > 100 dặm/giờ (Phi lý): 732 chuyến
• Số chuyến xe chạy liên tục trên 24 tiếng (Lỗi quên tắt đồng hồ): 35 chuyến

• Mức cước trần (Upper bound theo chuẩn 3-IQR) là khoảng: $74.40
• Số chuyến xe có cước phí siêu đắt (Hơn $300): 442 chuyến

--- TOP 5 CHUYẾN XE CÓ CƯỚC ĐẮT NHẤT ---


,tpep_pickup_datetime,trip_distance,trip_duration_minutes,speed_mph,fare_amount,total_amount
2127025,2026-01-25 00:43:10,48.65,3525.700000,0.827921,2555.2,2560.20
1230759,2026-01-15 13:56:22,0.00,0.000000,NaN,2500.0,2500.00
1608067,2026-01-19 17:09:00,0.00,0.083333,0.000000,990.0,991.00
2035988,2026-01-24 00:18:59,9.03,24.883333,21.773610,900.0,904.25
2167312,2026-01-26 17:27:28,128.05,175.266667,43.836059,899.0,911.71


In [9]:
# Danh sách các biến liên tục cần rà soát Outlier
continuous_vars = [
    'trip_distance', 
    'trip_duration_minutes', 
    'speed_mph', 
    'fare_amount', 
    'tip_amount', 
    'total_amount'
]

print("--- BÁO CÁO NGOẠI LAI (OUTLIERS) CHO CÁC BIẾN LIÊN TỤC ---")

for col in continuous_vars:
    # Bỏ qua các giá trị NaN để tính toán chính xác
    valid_data = df[col].dropna()
    
    # Tính Q1, Q3 và IQR
    Q1 = valid_data.quantile(0.25)
    Q3 = valid_data.quantile(0.75)
    IQR = Q3 - Q1
    
    # Tính Upper Bound (Ta dùng 3*IQR cho taxi thay vì 1.5*IQR để nới lỏng mức trần, tránh xóa nhầm các chuyến đi sân bay xa)
    upper_bound = Q3 + 3 * IQR
    
    # Đếm số lượng Outlier
    outliers_count = df[df[col] > upper_bound].shape[0]
    outliers_pct = (outliers_count / len(df)) * 100
    
    print(f"\n🔹 Cột: {col}")
    print(f"  - Mức trần (Upper Bound): {upper_bound:.2f}")
    print(f"  - Số lượng ngoại lai (> Trần): {outliers_count:,} dòng ({outliers_pct:.2f}%)")


--- BÁO CÁO NGOẠI LAI (OUTLIERS) CHO CÁC BIẾN LIÊN TỤC ---

🔹 Cột: trip_distance
  - Mức trần (Upper Bound): 11.92
  - Số lượng ngoại lai (> Trần): 199,302 dòng (5.35%)

🔹 Cột: trip_duration_minutes
  - Mức trần (Upper Bound): 60.60
  - Số lượng ngoại lai (> Trần): 67,059 dòng (1.80%)

🔹 Cột: speed_mph
  - Mức trần (Upper Bound): 31.56
  - Số lượng ngoại lai (> Trần): 69,073 dòng (1.85%)

🔹 Cột: fare_amount
  - Mức trần (Upper Bound): 74.40
  - Số lượng ngoại lai (> Trần): 48,258 dòng (1.30%)

🔹 Cột: tip_amount
  - Mức trần (Upper Bound): 14.84
  - Số lượng ngoại lai (> Trần): 88,305 dòng (2.37%)

🔹 Cột: total_amount
  - Mức trần (Upper Bound): 84.32
  - Số lượng ngoại lai (> Trần): 136,526 dòng (3.67%)


In [10]:
print("--- KIỂM TRA NGOẠI LAI DỰA TRÊN LOGIC THỰC TẾ ---")

# 1. Passenger Count (Số lượng khách tối đa của 1 xe Van thường là 9)
crazy_passengers = df[df['passenger_count'] > 9].shape[0]
print(f"• Số chuyến xe chở hơn 9 người (Nhồi nhét hoặc lỗi gõ phím): {crazy_passengers:,} chuyến")

# 2. Tiền qua trạm thu phí (Tolls Amount)
# Rất hiếm khi tiền cầu đường cho 1 cuốc taxi vượt quá $50 - $100
crazy_tolls = df[df['tolls_amount'] > 100].shape[0]
print(f"• Số chuyến có tiền qua trạm thu phí đắt hơn $100: {crazy_tolls:,} chuyến")

# 3. Phụ phí khác (Extra)
# Thường chỉ là $0.5 (đêm) hoặc $1.0 (giờ cao điểm)
crazy_extra = df[df['extra'] > 20].shape[0]
print(f"• Số chuyến có phụ phí giờ cao điểm/đêm > $20: {crazy_extra:,} chuyến")

# In thử top 5 chuyến chở nhiều khách nhất xem hệ thống ghi nhận mấy người
print("\n--- TOP 5 CHUYẾN XE CHỞ NHIỀU KHÁCH NHẤT ---")
df[['tpep_pickup_datetime', 'passenger_count', 'trip_distance', 'total_amount']].sort_values(by='passenger_count', ascending=False).head(5)


--- KIỂM TRA NGOẠI LAI DỰA TRÊN LOGIC THỰC TẾ ---
• Số chuyến xe chở hơn 9 người (Nhồi nhét hoặc lỗi gõ phím): 0 chuyến


• Số chuyến có tiền qua trạm thu phí đắt hơn $100: 5 chuyến
• Số chuyến có phụ phí giờ cao điểm/đêm > $20: 0 chuyến

--- TOP 5 CHUYẾN XE CHỞ NHIỀU KHÁCH NHẤT ---


,tpep_pickup_datetime,passenger_count,trip_distance,total_amount
212630,2026-01-03 21:27:58,9.0,0.00,98.75
1631806,2026-01-20 02:50:03,8.0,0.00,101.70
1231152,2026-01-15 13:38:23,8.0,10.95,102.00
2320189,2026-01-28 17:06:15,8.0,17.36,94.75
870138,2026-01-11 14:16:24,8.0,0.00,89.50


In [11]:
print("--- KIỂM TRA LỖI LOGIC: TỔNG TIỀN BỊ NHỎ HƠN THÀNH PHẦN ---")

# Kiểm tra Tổng tiền < Cước gốc
invalid_fare_logic = df[df['total_amount'] < df['fare_amount']].shape[0]
print(f"• Số chuyến có Tổng tiền nhỏ hơn Cước gốc: {invalid_fare_logic:,} chuyến")

# Kiểm tra Tổng tiền < Tiền Tip
invalid_tip_logic = df[df['total_amount'] < df['tip_amount']].shape[0]
print(f"• Số chuyến có Tổng tiền nhỏ hơn cả Tiền Tip: {invalid_tip_logic:,} chuyến")

# Kiểm tra Tổng tiền < Tiền cầu đường (Tolls)
invalid_tolls_logic = df[df['total_amount'] < df['tolls_amount']].shape[0]
print(f"• Số chuyến có Tổng tiền nhỏ hơn cả Tiền qua trạm thu phí: {invalid_tolls_logic:,} chuyến")

# Hiển thị thử các chuyến bị lỗi Tổng tiền < Cước gốc để xem chuyện gì xảy ra
print("\n--- XEM THỬ CÁC CHUYẾN CÓ TỔNG TIỀN < CƯỚC GỐC ---")
cols_to_check = ['fare_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'payment_type']
df[df['total_amount'] < df['fare_amount']][cols_to_check].head(5)


--- KIỂM TRA LỖI LOGIC: TỔNG TIỀN BỊ NHỎ HƠN THÀNH PHẦN ---
• Số chuyến có Tổng tiền nhỏ hơn Cước gốc: 42,691 chuyến
• Số chuyến có Tổng tiền nhỏ hơn cả Tiền Tip: 40,007 chuyến
• Số chuyến có Tổng tiền nhỏ hơn cả Tiền qua trạm thu phí: 40,004 chuyến

--- XEM THỬ CÁC CHUYẾN CÓ TỔNG TIỀN < CƯỚC GỐC ---


,fare_amount,mta_tax,tip_amount,tolls_amount,total_amount,payment_type
74,-5.1,-0.5,0.0,0.0,-10.10,4
248,-22.6,-0.5,0.0,0.0,-28.35,4
368,-3.0,-0.5,0.0,0.0,-8.75,4
428,-3.0,-0.5,0.0,0.0,-8.75,4
649,-22.6,-0.5,0.0,0.0,-28.35,2


### Kiểm tra logic nghiệp vụ

In [12]:
print("--- KIỂM TRA LOGIC NGHIỆP VỤ (BUSINESS LOGIC) ---")

# 1. Thanh toán Tiền mặt nhưng lại ghi nhận Tiền Tip (Boa) qua thẻ
# Theo luật của TLC, nếu khách trả tiền mặt (payment_type = 2), hệ thống sẽ KHÔNG ghi nhận tiền tip. Tiền tip chỉ ghi nhận khi quẹt thẻ.
cash_with_tip = df[(df['payment_type'] == 2) & (df['tip_amount'] > 0)].shape[0]
print(f"• Số chuyến trả Tiền mặt nhưng vẫn ghi nhận Tiền Tip: {cash_with_tip:,} chuyến")

# 2. Đi sân bay JFK với giá siêu rẻ
# RatecodeID = 2 là cuốc xe đi sân bay JFK. Giá cước của chuyến này bị Fix cứng (Flat Fare) là $70.
jfk_cheap = df[(df['RatecodeID'] == 2) & (df['fare_amount'] < 20)].shape[0]
print(f"• Số chuyến đi sân bay JFK nhưng giá cước < $20 (Mâu thuẫn giá cố định): {jfk_cheap:,} chuyến")

# 3. Mâu thuẫn Quãng đường và Cước phí
# Giá cước khởi điểm của Taxi NY là $3.00, cứ mỗi 1 dặm thêm khoảng $2.5 - $3.5. 
# Việc đi quá xa (> 5 dặm) nhưng giá cước lại chỉ bằng giá mở cửa (<= $5) là vi phạm vật lý.
far_but_cheap = df[(df['trip_distance'] > 5) & (df['fare_amount'] <= 5)].shape[0]
print(f"• Số chuyến đi xa (> 5 dặm) nhưng giá rẻ bèo (<= $5): {far_but_cheap:,} chuyến")

# 4. Cuốc xe "Tàng hình" (Unknown Zones)
# Theo bản đồ taxi, LocationID = 264 và 265 là các khu vực "Không xác định" (NV / Unknown). 
# Đây thường là lỗi rớt mạng GPS. Cần cảnh giác vì chúng không thể lên bản đồ.
unknown_zones = df[(df['pu_LocationID'].isin([264, 265])) | (df['do_LocationID'].isin([264, 265]))].shape[0]
print(f"• Số chuyến xe bắt hoặc thả khách ở khu vực Không xác định: {unknown_zones:,} chuyến")



--- KIỂM TRA LOGIC NGHIỆP VỤ (BUSINESS LOGIC) ---
• Số chuyến trả Tiền mặt nhưng vẫn ghi nhận Tiền Tip: 24 chuyến
• Số chuyến đi sân bay JFK nhưng giá cước < $20 (Mâu thuẫn giá cố định): 2,465 chuyến


• Số chuyến đi xa (> 5 dặm) nhưng giá rẻ bèo (<= $5): 12,886 chuyến
• Số chuyến xe bắt hoặc thả khách ở khu vực Không xác định: 25,251 chuyến


## 2. Tiền xử lí dữ liệu


In [13]:
print("--- BẮT ĐẦU LÀM SẠCH DỮ LIỆU (LOẠI BỎ LỖI LOGIC TRƯỚC) ---")
print(f"Số dòng ban đầu: {len(df):,}")

# Xác định ranh giới thời gian (Chỉ lấy đúng Tháng 01/2026)
start_date = pd.to_datetime('2026-01-01')
end_date = pd.to_datetime('2026-02-01')

# 1. BỘ LỌC LOẠI BỎ CÁI SAI (DÙNG DẤU `~` ĐỂ BẢO TOÀN NULL)
import gc
gc.collect()
# XÓA LỖI LOGIC TUẦN TỰ ĐỂ TRÁNH TRÀN RAM (MemoryError)
df.drop(df[(df['tpep_pickup_datetime'] < start_date) | (df['tpep_pickup_datetime'] >= end_date)].index, inplace=True, errors='ignore')
df.drop(df[df['passenger_count'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['trip_distance'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['fare_amount'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] < df['fare_amount']].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] < df['tip_amount']].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] < df['tolls_amount']].index, inplace=True, errors='ignore')
if 'speed_mph' in df.columns: df.drop(df[df['speed_mph'] > 80].index, inplace=True, errors='ignore')
df.drop(df[df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime']].index, inplace=True, errors='ignore')
df_clean = df

print(f"Số dòng sau khi Xóa các Lỗi Logic: {len(df_clean):,}")

# 2. ĐIỀN KHUYẾT MISSING (THỰC HIỆN SAU CÙNG)
# Điền Số lượng khách và Mã cước (Kiểu số)
df_clean['passenger_count'] = df_clean['passenger_count'].fillna(1.0)
df_clean['RatecodeID'] = df_clean['RatecodeID'].fillna(99.0)

# Điền các cột Thuế, Phí (Kiểu số)
cols_to_fill_zero = ['Airport_fee', 'congestion_surcharge']
df_clean[cols_to_fill_zero] = df_clean[cols_to_fill_zero].fillna(0.0)

# Điền các cột Chữ (Kiểu Chuỗi/String) để Parquet không bị lỗi
df_clean['store_and_fwd_flag'] = df_clean['store_and_fwd_flag'].fillna('N')
df_clean['ratecode_name'] = df_clean['ratecode_name'].fillna('Unknown')



--- BẮT ĐẦU LÀM SẠCH DỮ LIỆU (LOẠI BỎ LỖI LOGIC TRƯỚC) ---
Số dòng ban đầu: 3,724,889
Số dòng sau khi Xóa các Lỗi Logic: 3,499,217


In [14]:
print("=== BÁO CÁO NGHIỆM THU DỮ LIỆU SẠCH (FINAL INSPECTION) ===")

# Bài Test 1: Kiểm tra Khuyết thiếu (Missing Values)
# Đảm bảo các cột quan trọng không còn bất kỳ chữ NaN nào
missing_final = df_clean.isnull().sum()
print("\n1. SỐ LƯỢNG MISSING VALUES CÒN LẠI:")
print(missing_final[missing_final > 0]) # Chỉ in ra những cột còn lỗi
if missing_final.sum() == 0 or missing_final[['passenger_count', 'fare_amount']].sum() == 0:
    print("=> PASS! Dữ liệu đã được lấp đầy hoàn hảo.")

# Bài Test 2: Kiểm tra Logic Tối thiểu & Tối đa (Sanity Check)
# Dùng hàm describe() để soi các giá trị Min, Max xem có lọt lưới phần tử âm nào không
print("\n2. KIỂM TRA LOGIC TOÁN HỌC (MIN / MAX):")
sanity_check = df_clean[['trip_distance', 'fare_amount', 'passenger_count', 'speed_mph']].describe().loc[['min', 'max']]
print(sanity_check)
print("=> Bấm soi kỹ hàng 'min': Tất cả phải > 0.")
print("=> Bấm soi kỹ hàng 'max': Tốc độ (speed_mph) không được vượt quá 80.")

# Bài Test 3: Báo cáo tỷ lệ rớt (Retention Rate)
retention_rate = len(df_clean) / len(df) * 100
print(f"\n3. TỶ LỆ DỮ LIỆU GIỮ LẠI:")
print(f"• Dòng ban đầu: {len(df):,}")
print(f"• Dòng sạch: {len(df_clean):,}")
print(f"=> Đã giữ lại: {retention_rate:.2f}% tập dữ liệu.")


=== BÁO CÁO NGHIỆM THU DỮ LIỆU SẠCH (FINAL INSPECTION) ===

1. SỐ LƯỢNG MISSING VALUES CÒN LẠI:
pu_Borough           662
pu_Zone             3970
pu_service_zone     4632
do_Borough         14538
do_Zone             5105
do_service_zone    19643
dtype: int64
=> PASS! Dữ liệu đã được lấp đầy hoàn hảo.

2. KIỂM TRA LOGIC TOÁN HỌC (MIN / MAX):
     trip_distance  fare_amount  passenger_count  speed_mph
min           0.01         0.01              1.0   0.000342
max         257.24      2555.20              8.0  79.857347
=> Bấm soi kỹ hàng 'min': Tất cả phải > 0.
=> Bấm soi kỹ hàng 'max': Tốc độ (speed_mph) không được vượt quá 80.

3. TỶ LỆ DỮ LIỆU GIỮ LẠI:
• Dòng ban đầu: 3,499,217
• Dòng sạch: 3,499,217
=> Đã giữ lại: 100.00% tập dữ liệu.


In [15]:
import numpy as np
import pandas as pd

print("--- BẮT ĐẦU LÀM SẠCH DỮ LIỆU ---")

start_date = pd.to_datetime('2026-01-01')
end_date = pd.to_datetime('2026-02-01')

# ==========================================
# BƯỚC 1 & 2: LỌC BỎ LỖI VÀ ĐIỀN KHUYẾT
# ==========================================
import gc
gc.collect()
# XÓA LỖI LOGIC TUẦN TỰ ĐỂ TRÁNH TRÀN RAM (MemoryError)
df.drop(df[(df['tpep_pickup_datetime'] < start_date) | (df['tpep_pickup_datetime'] >= end_date)].index, inplace=True, errors='ignore')
df.drop(df[df['passenger_count'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['trip_distance'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['fare_amount'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] <= 0].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] < df['fare_amount']].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] < df['tip_amount']].index, inplace=True, errors='ignore')
df.drop(df[df['total_amount'] < df['tolls_amount']].index, inplace=True, errors='ignore')
if 'speed_mph' in df.columns: df.drop(df[df['speed_mph'] > 80].index, inplace=True, errors='ignore')
df.drop(df[df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime']].index, inplace=True, errors='ignore')
df_clean = df

# Điền khuyết
df_clean['passenger_count'] = df_clean['passenger_count'].fillna(1.0)
df_clean['RatecodeID'] = df_clean['RatecodeID'].fillna(99.0)
df_clean[['Airport_fee', 'congestion_surcharge']] = df_clean[['Airport_fee', 'congestion_surcharge']].fillna(0.0)
df_clean['store_and_fwd_flag'] = df_clean['store_and_fwd_flag'].fillna('N')
df_clean['ratecode_name'] = df_clean['ratecode_name'].fillna('Unknown')
geo_cols = ['pu_Borough', 'pu_Zone', 'pu_service_zone', 'do_Borough', 'do_Zone', 'do_service_zone']
df_clean[geo_cols] = df_clean[geo_cols].fillna('Unknown')


# ==========================================
# BƯỚC 3: KIỂM TRA LẠI LẦN CUỐI (SANITY CHECK)
# ==========================================
print("\n=== BÁO CÁO NGHIỆM THU TRƯỚC KHI XUẤT FILE ===")

# Test 1: Quét sạch Missing Values
missing_final = df_clean.isnull().sum()
missing_cols = missing_final[missing_final > 0]
if len(missing_cols) == 0:
    print("✅ TEST 1 (MISSING VALUES): Tuyệt vời! Không còn một ô trống nào.")
else:
    print("❌ TEST 1 FAILED: Cảnh báo! Vẫn còn dữ liệu Missing:")
    print(missing_cols)

# Test 2: Quét lỗi Logic (Số âm, vô lý)
print("\n✅ TEST 2 (LOGIC TOÁN HỌC - Bảng tóm tắt Min/Max):")
sanity_check = df_clean[['trip_distance', 'fare_amount', 'passenger_count', 'speed_mph']].describe().loc[['min', 'max']]
print(sanity_check)

print(f"\n=> Tỷ lệ dữ liệu sạch giữ lại được: {(len(df_clean) / len(df) * 100):.1f}% (Số dòng: {len(df_clean):,})")

# ==========================================
# BƯỚC 4: XUẤT FILE SAU KHI ĐÃ ĐẠT CHUẨN
# ==========================================
if len(missing_cols) == 0 and sanity_check.loc['min', 'fare_amount'] > 0:
    print("\n=== TIẾN HÀNH XUẤT FILE ===")
    clean_data_path = r'../../data/yellow_tripdata_2026-01_CLEANED.parquet'
    df_clean.to_parquet(clean_data_path, index=False)
    print(f"🎉 Đã lưu thành công dữ liệu SIÊU SẠCH ra file: {clean_data_path}")
else:
    print("\n⚠️ KHÔNG XUẤT FILE: Dữ liệu vẫn còn lỗi, vui lòng kiểm tra lại bảng Report ở trên!")


--- BẮT ĐẦU LÀM SẠCH DỮ LIỆU ---

=== BÁO CÁO NGHIỆM THU TRƯỚC KHI XUẤT FILE ===
✅ TEST 1 (MISSING VALUES): Tuyệt vời! Không còn một ô trống nào.

✅ TEST 2 (LOGIC TOÁN HỌC - Bảng tóm tắt Min/Max):
     trip_distance  fare_amount  passenger_count  speed_mph
min           0.01         0.01              1.0   0.000342
max         257.24      2555.20              8.0  79.857347

=> Tỷ lệ dữ liệu sạch giữ lại được: 100.0% (Số dòng: 3,499,217)

=== TIẾN HÀNH XUẤT FILE ===
🎉 Đã lưu thành công dữ liệu SIÊU SẠCH ra file: ../../data/yellow_tripdata_2026-01_CLEANED.parquet
